# ChuckleNet WavLM + Prosody Extraction v6

**Fixed: Uses existing MP3 files from Drive + uploads utterances**

- Audio from `chuckle_audio/` and `chuckle_audio_all/` (MP3, NOT M4A)
- 768-dim WavLM embeddings (attention pooling)
- 21-dim prosody features
- Train/Val/Test split by comedian
- Progress every 100 utterances
- Checkpoint every 1000 utterances


## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

## Step 2: Install Packages

In [ ]:
!pip install -q transformers librosa scikit-learn
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Packages installed!')

## Step 3: Upload utterances_clean.jsonl to Drive

**FIRST: Run this cell to upload the local utterances file**

Then copy the path it outputs and use it in Step 4.

In [ ]:
from google.colab import files
import os

# Upload the utterances file
print('📤 Upload utterances_clean.jsonl from your local machine:')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'✅ Uploaded: {filename} ({len(uploaded[filename])} bytes)')
    UTTERANCES_LOCAL = filename

# Copy to Drive for persistence
!cp "$UTTERANCES_LOCAL" /content/gdrive/MyDrive/utterances_clean.jsonl
print('✅ Copied to /content/gdrive/MyDrive/utterances_clean.jsonl')

## Step 4: Load Utterances & Build Audio Map

In [ ]:
import json
from pathlib import Path

# Load utterances from Drive
UTTERANCES_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
utterances = []
with open(UTTERANCES_FILE, 'r') as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'📋 Loaded {len(utterances)} utterances')

# Build audio file map from both folders (MP3!)
audio_base_1 = Path('/content/gdrive/MyDrive/chuckle_audio')
audio_base_2 = Path('/content/gdrive/MyDrive/chuckle_audio_all')

audio_files = {}
for p in audio_base_1.glob('*.mp3'):
    audio_files[p.stem] = str(p)
for p in audio_base_2.glob('**/*.mp3'):
    audio_files[p.stem] = str(p)

print(f'🎵 Found {len(audio_files)} MP3 audio files')

# Filter to only utterances with audio
valid_utterances = [u for u in utterances if u['video_id'] in audio_files]
print(f'✅ Valid utterances with audio: {len(valid_utterances)}')

# Sample check
if len(valid_utterances) > 0:
    print(f'   Sample: {valid_utterances[0]["video_id"]} -> {audio_files.get(valid_utterances[0]["video_id"], "NOT FOUND")}')

## Step 5: Define 21-dim Prosody Extractor

In [ ]:
import numpy as np
import librosa

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features from audio segment."""
    features = []
    
    # 1. F0 (pitch) features - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0, 0, 0, 0, 0])
    
    # 2. Energy features - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms),
        np.std(rms),
        np.max(rms),
        np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # 3. Duration features - 2 dims
    duration = len(y) / sr
    speech_rate = np.sum(voiced_flag) / duration if duration > 0 else 0
    features.extend([duration, speech_rate])
    
    # 4. Spectral features - 5 dims
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=3)
        mfcc_delta = librosa.feature.delta(mfccs)
        spectral_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features.extend([
            np.mean(mfccs[0]),
            np.mean(mfccs[1]),
            np.mean(mfccs[2]),
            np.mean(mfcc_delta),
            np.mean(spectral_cent) / sr * 1000
        ])
    except:
        features.extend([0, 0, 0, 0, 0])
    
    # 5. Voice quality features - 4 dims
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(zcr),
        np.std(f0_clean) / np.mean(f0_clean) if len(f0_clean) > 1 and np.mean(f0_clean) > 0 else 0,
        np.std(rms) / (np.mean(rms) + 1e-8),
        np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
    ])
    
    return np.array(features, dtype=np.float32)  # 21 dims

# Test
test_y = np.random.randn(16000)
test_pros = extract_prosody_21dim(test_y, 16000)
print(f'✅ Prosody extractor: {test_pros.shape[0]} dims')

## Step 6: Download Wav2Vec2 Model

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch

MODEL_NAME = 'facebook/wav2vec2-large-xlsr-53'

print('📥 Downloading Wav2Vec2 model...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
wav2vec = Wav2Vec2Model.from_pretrained(MODEL_NAME)
wav2vec = wav2vec.to('cuda')
wav2vec.eval()
print('✅ Model loaded!')

## Step 7: Create Train/Val/Test Split (by comedian)

In [ ]:
def get_split(video_id):
    """Determine split based on video_id (comedian identity)."""
    test_comedians = ['russell', 'burr', 'chappelle', 'bourdain', 'gal', 'CK louis']
    val_comedians = ['rock', 'seinfeld', 'ching', 'schmidt', 'norton']
    
    video_lower = video_id.lower()
    
    for comedian in test_comedians:
        if comedian in video_lower:
            return 'test'
    for comedian in val_comedians:
        if comedian in video_lower:
            return 'val'
    return 'train'

# Assign splits
for u in valid_utterances:
    u['split'] = get_split(u['video_id'])

train_count = sum(1 for u in valid_utterances if u['split'] == 'train')
val_count = sum(1 for u in valid_utterances if u['split'] == 'val')
test_count = sum(1 for u in valid_utterances if u['split'] == 'test')

print(f'📊 Split distribution:')
print(f'   Train: {train_count} ({train_count/len(valid_utterances)*100:.1f}%)')
print(f'   Val:   {val_count} ({val_count/len(valid_utterances)*100:.1f}%)')
print(f'   Test:  {test_count} ({test_count/len(valid_utterances)*100:.1f}%)')

## Step 8: Extract Features (WITH PROGRESS)

In [ ]:
import numpy as np
import librosa
import time
import os
import json

# Config
SR = 16000
BATCH_SIZE = 32
CHECKPOINT_DIR = '/content/gdrive/MyDrive/wavlm_extraction_checkpoints_v6'
CHECKPOINT_INTERVAL = 1000

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Check for existing checkpoint
checkpoint_file = os.path.join(CHECKPOINT_DIR, 'extraction_checkpoint.json')
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    start_idx = checkpoint['last_idx'] + 1
    train_emb, train_pros, train_lbl = checkpoint.get('train_emb', [])[-10000:], checkpoint.get('train_pros', [])[-10000:], checkpoint.get('train_lbl', [])[-10000:]
    val_emb, val_pros, val_lbl = checkpoint.get('val_emb', []), checkpoint.get('val_pros', []), checkpoint.get('val_lbl', [])
    test_emb, test_pros, test_lbl = checkpoint.get('test_emb', []), checkpoint.get('test_pros', []), checkpoint.get('test_lbl', [])
    print(f'📂 Resuming from checkpoint at index {start_idx}')
    print(f'   Train: {len(train_emb)}, Val: {len(val_emb)}, Test: {len(test_emb)}')
else:
    start_idx = 0
    train_emb, train_pros, train_lbl = [], [], []
    val_emb, val_pros, val_lbl = [], [], []
    test_emb, test_pros, test_lbl = [], [], []
    print('🆕 Starting fresh extraction')

total = len(valid_utterances)
t0 = time.time()
skipped = 0

# Process in batches
for batch_start in range(start_idx, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch = valid_utterances[batch_start:batch_end]
    
    for u in batch:
        audio_path = audio_files.get(u['video_id'])
        if audio_path is None:
            skipped += 1
            continue
        
        try:
            # Load audio (offset + duration for this utterance)
            y, sr = librosa.load(audio_path, sr=SR, mono=True, 
                               offset=u['start'], 
                               duration=u['end']-u['start'])
            
            # Skip very short
            if len(y) < SR * 0.1:
                continue
            
            # Get Wav2Vec2 features
            inputs = processor(y, sampling_rate=SR, return_tensors='pt', padding=True)
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = wav2vec(**inputs)
            
            # Attention pooling
            hidden = outputs.last_hidden_state
            attn_weights = torch.softmax(torch.nn.Linear(768, 1)(hidden).squeeze(-1), dim=-1)
            emb = torch.bmm(attn_weights.unsqueeze(1), hidden).squeeze(1).squeeze(0).cpu().numpy()
            
            # Extract 21-dim prosody
            prosody = extract_prosody_21dim(y, sr)
            
            # Store by split
            split = u['split']
            if split == 'train':
                train_emb.append(emb)
                train_pros.append(prosody)
                train_lbl.append(u['label'])
            elif split == 'val':
                val_emb.append(emb)
                val_pros.append(prosody)
                val_lbl.append(u['label'])
            else:
                test_emb.append(emb)
                test_pros.append(prosody)
                test_lbl.append(u['label'])
            
        except Exception as e:
            # Skip problematic files silently
            skipped += 1
            continue
    
    # Progress update
    current_idx = batch_end
    elapsed = time.time() - t0
    rate = current_idx / elapsed if elapsed > 0 else 0
    eta = (total - current_idx) / rate / 60 if rate > 0 else 0
    total_processed = len(train_lbl) + len(val_lbl) + len(test_lbl)
    pos_total = sum(train_lbl) + sum(val_lbl) + sum(test_lbl)
    pos_rate = pos_total / max(total_processed, 1) * 100
    
    print(f'📊 {current_idx}/{total} ({(current_idx/total*100):.1f}%) | '
          f'{rate:.1f} utt/s | ETA: {eta:.1f}min | '
          f'T:{len(train_lbl)} V:{len(val_lbl)} Te:{len(test_lbl)} | '
          f'Pos:{pos_rate:.1f}% | Skip:{skipped}')
    
    # Save checkpoint
    if current_idx % CHECKPOINT_INTERVAL == 0 or current_idx >= total:
        checkpoint_data = {
            'last_idx': current_idx - 1,
            'train_emb': train_emb[-10000:],
            'train_pros': train_pros[-10000:],
            'train_lbl': train_lbl[-10000:],
            'val_emb': val_emb,
            'val_pros': val_pros,
            'val_lbl': val_lbl,
            'test_emb': test_emb,
            'test_pros': test_pros,
            'test_lbl': test_lbl,
            'timestamp': time.time()
        }
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f)
        print(f'💾 Checkpoint saved')

print(f'\n✅ Extraction complete!')
print(f'   Train: {len(train_emb)} samples')
print(f'   Val:   {len(val_emb)} samples')
print(f'   Test:  {len(test_emb)} samples')
print(f'   Skipped: {skipped}')

## Step 9: Save Final Results

In [ ]:
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_555_v6'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save as NPZ
np.savez(
    os.path.join(OUTPUT_DIR, 'train.npz'),
    wavlm=np.array(train_emb, dtype=np.float32),
    prosody=np.array(train_pros, dtype=np.float32),
    labels=np.array(train_lbl, dtype=np.int32)
)
np.savez(
    os.path.join(OUTPUT_DIR, 'val.npz'),
    wavlm=np.array(val_emb, dtype=np.float32),
    prosody=np.array(val_pros, dtype=np.float32),
    labels=np.array(val_lbl, dtype=np.int32)
)
np.savez(
    os.path.join(OUTPUT_DIR, 'test.npz'),
    wavlm=np.array(test_emb, dtype=np.float32),
    prosody=np.array(test_pros, dtype=np.float32),
    labels=np.array(test_lbl, dtype=np.int32)
)

print(f'✅ Results saved to {OUTPUT_DIR}')
print(f'   Train: {len(train_emb)} x WavLM{list(np.array(train_emb).shape[1:]) if len(train_emb) > 0 else "?"} + Prosody{list(np.array(train_pros).shape[1:]) if len(train_pros) > 0 else "?"}')
print(f'   Val:   {len(val_emb)} samples')
print(f'   Test:  {len(test_emb)} samples')